# SIRAP — Anotación y evaluación de video con referencia por intervalos

Este notebook contiene la guía y el código para crear, revisar y usar `video_ground_truth.csv`. La referencia tiene una fila por cada pasada visible y un intervalo inclusivo de cuadros en el que la placa se puede leer. La placa se transcribe mirando el video; no se copia del OCR.

## Qué permite hacer

1. Abrir el video que se va a anotar y revisar cuadros con número y tiempo.
2. Registrar una fila por placa y pasada, validar límites y formato, y guardar una copia de trabajo.
3. Cargar la referencia publicada del repositorio si se desea evaluarla sin editar.
4. Comparar un CSV de lecturas producido por el pipeline SIRAP con los intervalos y calcular métricas con alcance explícito.

La evaluación compara una lectura con una placa real solo cuando el `frame_index` de esa lectura cae dentro de un intervalo anotado. Las lecturas fuera de intervalos quedan sin clasificar; por eso estas métricas no describen por sí solas todos los vehículos del video. Para una evaluación completa hay que revisar también el video buscando vehículos que el detector no registró.

**Requisito de evaluación:** usa el video original al que corresponde la referencia. El clip de demostración sirve para aprender a usar la guía, pero no se debe puntuar contra etiquetas de otro video.


## 1. Entorno y archivos

Puedes abrir este notebook en Google Colab. Se clona el repositorio del proyecto para acceder a la referencia publicada y al clip demostrativo. Para una evaluación propia, sube el video a `/content/SIRAP_workspace/video_camiones.mp4`. Si se usa otro nombre o ubicación, cambia `SOURCE_VIDEO_PATH` en la celda de configuración.

La evaluación del pipeline requiere además el CSV de decisiones generado por la sección 6.9 del notebook principal. El notebook descargará automáticamente desde GitHub el archivo `results/metrics/YOLO11_yolo11s_completo_decisions_log.csv`.


In [52]:
from pathlib import Path
import subprocess
import shutil
import re
import cv2
import numpy as np
import pandas as pd
from IPython.display import display

REPOSITORY_URL = "https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git"
REPO_ROOT = Path("/content/SIRAP_USA_ECUADOR")
if not (REPO_ROOT / ".git").exists():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_ROOT)], check=True)

WORK_DIR = Path("/content/SIRAP_workspace")
WORK_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_VIDEO_PATH = WORK_DIR / "video_camiones.mp4"
DEMO_VIDEO_PATH = REPO_ROOT / "Videos_Imagenes_Prueba" / "video_01.mp4"
PUBLISHED_REFERENCE_PATH = REPO_ROOT / "blind_eval" / "video_ground_truth.csv"
WORKING_REFERENCE_PATH = WORK_DIR / "video_ground_truth.csv"


## 2. Seleccionar y revisar el video

La numeración empieza en cero. El tiempo aproximado de un cuadro es `frame_index / FPS`. Confirma que el video seleccionado es el mismo que se usó para generar el CSV de lecturas. Si el video original no está presente, la celda ofrece el clip de demostración y avisa que no corresponde a la referencia publicada.


In [53]:
if SOURCE_VIDEO_PATH.is_file():
    VIDEO_PATH = SOURCE_VIDEO_PATH
    print(f"Video de trabajo: {VIDEO_PATH}")
elif DEMO_VIDEO_PATH.is_file():
    VIDEO_PATH = DEMO_VIDEO_PATH
    print(f"No se encontró el video original en {SOURCE_VIDEO_PATH}.")
    print(f"Se usará el clip de demostración: {VIDEO_PATH}")
    print("No compares este clip con una referencia creada para otro video.")
else:
    raise FileNotFoundError(f"Sube el video a {SOURCE_VIDEO_PATH} o actualiza VIDEO_PATH.")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise ValueError(f"OpenCV no pudo abrir {VIDEO_PATH}")
VIDEO_FPS = float(cap.get(cv2.CAP_PROP_FPS))
VIDEO_TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
if VIDEO_FPS <= 0 or VIDEO_TOTAL_FRAMES <= 0:
    raise ValueError("No se pudo leer el FPS o el total de cuadros del video.")
print(f"{VIDEO_WIDTH}×{VIDEO_HEIGHT} px · {VIDEO_FPS:.3f} FPS · {VIDEO_TOTAL_FRAMES:,} cuadros · "
      f"{VIDEO_TOTAL_FRAMES / VIDEO_FPS / 60:.2f} min")


Video de trabajo: /content/SIRAP_workspace/video_camiones.mp4
1280×720 px · 30.000 FPS · 18,115 cuadros · 10.06 min


## 3. Navegar por cuadros

`show_frame(n)` muestra un cuadro concreto. `ver_cuadros(desde, hasta, n)` reparte una selección de cuadros en un intervalo para decidir dónde empieza y termina el tramo legible. Usa el mismo video y la misma numeración que el pipeline de SIRAP.


In [54]:
def read_frame(index: int):
    if not isinstance(index, (int, np.integer)) or not 0 <= int(index) < VIDEO_TOTAL_FRAMES:
        raise ValueError(f"El cuadro debe estar entre 0 y {VIDEO_TOTAL_FRAMES - 1}.")
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"No se pudo leer el cuadro {index}.")
    return frame

def show_frame(index: int, max_width: int = 1200):
    from matplotlib import pyplot as plt
    frame = read_frame(index)
    if max_width and frame.shape[1] > max_width:
        frame = cv2.resize(frame, (max_width, round(max_width * frame.shape[0] / frame.shape[1])),
                           interpolation=cv2.INTER_AREA)
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Cuadro {index} · {index / VIDEO_FPS:.2f} s")
    plt.axis("off")
    plt.show()

def ver_cuadros(desde: int, hasta: int, n: int = 6):
    if not (0 <= desde <= hasta < VIDEO_TOTAL_FRAMES):
        raise ValueError(f"Usa 0 ≤ desde ≤ hasta ≤ {VIDEO_TOTAL_FRAMES - 1}.")
    indices = np.unique(np.linspace(desde, hasta, max(1, n)).round().astype(int))
    from matplotlib import pyplot as plt
    fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 3.2), squeeze=False)
    for ax, index in zip(axes[0], indices):
        frame = read_frame(int(index))
        if frame.shape[1] > 640:
            frame = cv2.resize(frame, (640, round(640 * frame.shape[0] / frame.shape[1])),
                               interpolation=cv2.INTER_AREA)
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{index} · {index / VIDEO_FPS:.1f} s", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Ejemplo: show_frame(100) o ver_cuadros(100, 180, n=6)


## 4. Anotar las pasadas

Agrega una tupla `(placa, frame_start, frame_end)` por cada pasada. Si el mismo camión aparece en dos tramos distintos, crea dos filas. Si dos camiones se ven a la vez, crea una fila para cada placa. Los extremos son inclusivos: `frame_start` es el primer cuadro en el que la placa se puede leer y `frame_end` el último.

Escribe exactamente lo que se distingue en la placa. El código normaliza a `XXX-0000`; para una placa con tres dígitos añade un cero inicial al bloque numérico.


In [55]:
# Referencia publicada para el video original de la garita.
# Usa estas tuplas solo si el video cargado corresponde a esa referencia.
# No las uses para puntuar Videos_Imagenes_Prueba/video_01.mp4.
VIDEO_ANNOTATIONS = [
    ("GRW-5172", 472, 827),
    ("TAK-0131", 1489, 1916),
    ("PAA-2178", 1945, 2301),
    ("PAB-6510", 2380, 2771),
    ("PTD-0533", 3150, 3696),
    ("GBP-9871", 3812, 4076),
    ("GTA-6721", 4426, 4774),
    ("PZF-0329", 4775, 4899),
    ("ABT-8220", 5138, 5714),
    ("GBO-5174", 6142, 6389),
    ("GBN-2864", 6593, 7287),
    ("GBQ-2106", 8373, 9295),
    ("GLB-0246", 9814, 10547),
    ("GBQ-6255", 11304, 12214),
    ("GBQ-2801", 12784, 13430),
    ("PAB-6510", 13622, 14410),
    ("GPQ-0783", 14800, 15338),
    ("GBO-5174", 15804, 16473),
    ("PZF-0329", 16474, 16598),
    ("GBQ-3986", 17004, 18080),
]

def format_plate(value: str) -> str:
    text = re.sub(r"[^A-Z0-9]", "", str(value).upper())
    match = re.fullmatch(r"([A-Z]{3})([0-9]{3,4})", text)
    if not match:
        return ""
    letters, digits = match.groups()
    return f"{letters}-{digits.zfill(4)}"

def validate_annotations(rows):
    errors, warnings, clean = [], [], []
    for row_no, entry in enumerate(rows, start=1):
        if not isinstance(entry, (tuple, list)) or len(entry) != 3:
            errors.append(f"fila {row_no}: debe ser (placa, frame_start, frame_end)")
            continue
        plate, start, end = entry
        plate = format_plate(plate)
        if not plate:
            errors.append(f"fila {row_no}: placa inválida; revisa la transcripción")
        if isinstance(start, bool) or not isinstance(start, (int, np.integer)):
            errors.append(f"fila {row_no}: frame_start debe ser entero")
        if isinstance(end, bool) or not isinstance(end, (int, np.integer)):
            errors.append(f"fila {row_no}: frame_end debe ser entero")
        if isinstance(start, (int, np.integer)) and isinstance(end, (int, np.integer)):
            if not 0 <= start <= end < VIDEO_TOTAL_FRAMES:
                errors.append(f"fila {row_no}: se requiere 0 ≤ inicio ≤ fin ≤ {VIDEO_TOTAL_FRAMES - 1}")
        clean.append({"plate_gt": plate, "frame_start": start, "frame_end": end})
    for i, first in enumerate(clean):
        for j in range(i + 1, len(clean)):
            second = clean[j]
            valid_bounds = all(isinstance(x, (int, np.integer)) and not isinstance(x, bool)
                               for x in (first["frame_start"], first["frame_end"],
                                         second["frame_start"], second["frame_end"]))
            if (valid_bounds and first["plate_gt"] and first["plate_gt"] == second["plate_gt"]
                    and first["frame_start"] <= second["frame_end"]
                    and second["frame_start"] <= first["frame_end"]):
                warnings.append(f"filas {i+1} y {j+1}: intervalos de la misma placa se solapan; confirma si son pasadas distintas")
    if errors:
        raise ValueError("Corrige estas filas antes de guardar:\n- " + "\n- ".join(errors))
    return pd.DataFrame(clean), warnings

if VIDEO_ANNOTATIONS:
    annotation_df, annotation_warnings = validate_annotations(VIDEO_ANNOTATIONS)
    for warning in annotation_warnings:
        print("Aviso:", warning)
    display(annotation_df)
else:
    print("VIDEO_ANNOTATIONS está vacía. Completa las tuplas después de revisar los cuadros.")


,plate_gt,frame_start,frame_end
0,GRW-5172,472,827
1,TAK-0131,1489,1916
2,PAA-2178,1945,2301
3,PAB-6510,2380,2771
4,PTD-0533,3150,3696
5,GBP-9871,3812,4076
6,GTA-6721,4426,4774
7,PZF-0329,4775,4899
8,ABT-8220,5138,5714
9,GBO-5174,6142,6389


### Guardar la anotación

La salida se guarda en `/content/SIRAP_workspace/video_ground_truth.csv` con tres columnas obligatorias: `plate_gt`, `frame_start` y `frame_end`. Si ya existe una copia de trabajo, se conserva primero con sufijo `_anterior`. El CSV publicado del repositorio no se sobrescribe.


In [56]:
if VIDEO_ANNOTATIONS:
    annotation_df, annotation_warnings = validate_annotations(VIDEO_ANNOTATIONS)
    if WORKING_REFERENCE_PATH.exists():
        backup_path = WORKING_REFERENCE_PATH.with_name(WORKING_REFERENCE_PATH.stem + "_anterior.csv")
        shutil.copy2(WORKING_REFERENCE_PATH, backup_path)
        print("Copia previa:", backup_path)
    annotation_df.sort_values(["frame_start", "frame_end"]).to_csv(WORKING_REFERENCE_PATH, index=False)
    print(f"Guardadas {len(annotation_df)} pasadas en {WORKING_REFERENCE_PATH}")
else:
    print("No se guardó nada: completa VIDEO_ANNOTATIONS primero.")


Copia previa: /content/SIRAP_workspace/video_ground_truth_anterior.csv
Guardadas 20 pasadas en /content/SIRAP_workspace/video_ground_truth.csv


## 5. Cargar la referencia publicada o la copia de trabajo

La copia de trabajo tiene prioridad si existe. Para reproducir una evaluación con la referencia versionada en el repositorio, elige `USE_PUBLISHED_REFERENCE = True`. La referencia publicada contiene intervalos inclusivos asociados al video original de la garita.


In [57]:
USE_PUBLISHED_REFERENCE = True
REFERENCE_PATH = PUBLISHED_REFERENCE_PATH if USE_PUBLISHED_REFERENCE else WORKING_REFERENCE_PATH
if not REFERENCE_PATH.is_file():
    raise FileNotFoundError(f"No existe la referencia seleccionada: {REFERENCE_PATH}")

def load_video_reference(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    required = {"plate_gt", "frame_start", "frame_end"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")
    ref = pd.DataFrame({
        "plate_gt": raw["plate_gt"].map(format_plate),
        "frame_start": pd.to_numeric(raw["frame_start"], errors="coerce"),
        "frame_end": pd.to_numeric(raw["frame_end"], errors="coerce"),
    })
    bad = ((ref["plate_gt"] == "") | ref[["frame_start", "frame_end"]].isna().any(axis=1) |
           (ref["frame_start"] < 0) | (ref["frame_end"] < ref["frame_start"]) |
           (ref["frame_end"] >= VIDEO_TOTAL_FRAMES))
    if bad.any():
        raise ValueError("Filas inválidas o fuera del video actual:\n" + ref[bad].to_string(index=False))
    return ref.astype({"frame_start": int, "frame_end": int})

reference = load_video_reference(REFERENCE_PATH)
print(f"{REFERENCE_PATH} · {len(reference)} pasadas · {reference['plate_gt'].nunique()} placas distintas")
display(reference.head(10))


/content/SIRAP_USA_ECUADOR/blind_eval/video_ground_truth.csv · 20 pasadas · 17 placas distintas


,plate_gt,frame_start,frame_end
0,GRW-5172,472,827
1,TAK-0131,1489,1916
2,PAA-2178,1945,2301
3,PAB-6510,2380,2771
4,PTD-0533,3150,3696
5,GBP-9871,3812,4076
6,GTA-6721,4426,4774
7,PZF-0329,4775,4899
8,ABT-8220,5138,5714
9,GBO-5174,6142,6389


## 6. Evaluar el registro del pipeline

El notebook descarga desde GitHub el archivo `results/metrics/YOLO11_yolo11s_completo_decisions_log.csv`. Debe incluir `frame_index`, `normalized_text` y, para contar falsas autorizaciones, `status`. Cada fila del registro es una lectura del sistema. El código asocia una lectura solo con un intervalo que contenga su cuadro; si hay más de una placa visible en ese cuadro, asigna la referencia con texto más parecido al OCR.

- **EMR por lectura:** fracción de lecturas asociadas cuya placa normalizada coincide exactamente.
- **Recall de placas:** fracción de placas distintas de la referencia que el sistema leyó correctamente al menos una vez.
- **Lecturas sin referencia:** lecturas que caen fuera de todos los intervalos; se informan aparte y no entran en EMR.
- **Falsas autorizaciones en lecturas clasificables:** lecturas marcadas `AUTORIZADO` cuya placa real no aparece en la lista autorizada cargada. Este recuento no estima la tasa para el video completo.


In [58]:
from pathlib import Path
import requests

DECISIONS_CSV = WORK_DIR / "YOLO11_yolo11s_completo_decisions_log.csv"
DECISIONS_URL = (
    "https://raw.githubusercontent.com/eduardoceballos666/"
    "SIRAP_USA_ECUADOR/main/results/metrics/"
    "YOLO11_yolo11s_completo_decisions_log.csv"
)

response = requests.get(DECISIONS_URL, timeout=(15, 120))
response.raise_for_status()
DECISIONS_CSV.write_bytes(response.content)

print(f"Registro descargado desde GitHub: {DECISIONS_CSV}")


Registro descargado desde GitHub: /content/SIRAP_workspace/YOLO11_yolo11s_completo_decisions_log.csv


## 7. Interpretación y límites

La unidad de comparación es una lectura del registro dentro de un intervalo anotado. Si la placa no aparece en el registro, el intervalo queda como no detectado; el recall de placas refleja ese caso. Las pasadas sin una referencia confiable no deben incluirse como si fueran aciertos o errores. Revisa manualmente eventos que el detector no haya encontrado y documenta cualquier exclusión.

No combines métricas de ejecuciones distintas del pipeline ni cambies de video entre anotación y evaluación. Guarda junto al CSV la versión del modelo, configuración de umbrales, ruta o hash del video y fecha de ejecución para que los resultados puedan reproducirse.
